# 🧩 生成式AI應用開發｜第06週
## Structured Outputs 與資料抽取

> **執行環境：Google Colab** ｜ **延續第 3～5 週的 OpenAI Responses API**

前幾週的回覆都是「一段文字」，人要自己讀、自己找重點。本週讓模型直接回傳**符合指定格式的 JSON**，
可以放心用程式解析、驗證、存進資料庫。你會學到：

1. 為什麼純文字回覆難以穩定解析
2. 用 JSON Schema 描述你要的資料格式
3. 用 Pydantic 定義資料模型，簡化 Schema 撰寫
4. 抽取巢狀資料與清單（陣列）
5. 欄位驗證與抽取失敗的處理
6. 封裝成可重複使用的資料抽取函式

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  完成一個能把非結構化文字，轉換成可驗證 JSON 結構化資料的抽取工具。
</div>


> **教師版**：本檔保留完整參考答案，可用於課堂示範、投影講解或課後核對。


## 🎯 本週學習目標

| # | 能力 | 對應後續課程 |
|---|------|-------------|
| 1 | 理解為何純文字回覆難以穩定解析 | 所有資料抽取應用 |
| 2 | 用 JSON Schema 描述輸出格式 | 理解 Structured Outputs 原理 |
| 3 | 用 Pydantic 定義資料模型 | 簡化 Schema 撰寫、型別安全 |
| 4 | 抽取巢狀資料與清單 | 訂單、會議記錄等複雜資料 |
| 5 | 欄位驗證與失敗處理 | Week 11 RAG、實務資料清理 |
| 6 | 封裝可重複使用的抽取函式 | Week 7 Streamlit 應用整合 |


---
## 🧰 第一節：環境準備（延續第 3～5 週）

沿用先前的環境設定與 `ask_ai_safe()`。Colab 每次重開 runtime 都要重新安裝與設定金鑰。


In [ ]:
# 安裝 OpenAI Python SDK（Pydantic 已內建於 openai 套件相依）
!pip install openai --quiet

import os
import json
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

print("套件與模組準備完成")


### 🔐 設定 API Key

作法與第 3～5 週相同：在 Colab 左側 **Secrets** 面板新增 `OPENAI_API_KEY`，並打開存取權限。


In [ ]:
# 從 Colab Secrets 讀取 API key，並轉成環境變數
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("已從 Colab Secrets 載入 API key")
except Exception as e:
    print("非 Colab 環境或尚未設定 Secrets：", e)
    print("請改用環境變數或 .env 設定 OPENAI_API_KEY")


In [ ]:
# 建立 client、選模型，並沿用第 3 週的安全呼叫函式
client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")


def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    """單次呼叫 Responses API，回傳 (是否成功, 文字)。"""
    try:
        response = client.responses.create(
            model=model, instructions=role, input=user_input
        )
        return True, response.output_text
    except Exception as e:
        return False, f"API 呼叫失敗：{e}"


print(f"準備完成，使用模型：{MODEL}")


---
## 🔍 第二節：為什麼需要 Structured Outputs

請模型讀一段顧客評論，直接用文字說明重點，看看能不能穩定抽出姓名、評分與情緒。


In [ ]:
# 示範：純文字回覆很難穩定解析
review_text = "王小明給了這家餐廳五顆星，他說：「食物非常好吃，服務也很親切，會再來！」"

question = f"請閱讀以下評論，說明評論者姓名、給幾顆星、情緒是正面/中立/負面：\n\n{review_text}"
ok, answer = ask_ai_safe(question)
print(answer)

# 嘗試直接當 JSON 解析 —— 通常會失敗，因為這只是一段自然語言文字
try:
    data = json.loads(answer)
    print("解析成功：", data)
except json.JSONDecodeError as e:
    print("解析失敗：", e)


---
## 📐 第三節：用 JSON Schema 描述資料格式

JSON Schema 是一份「資料格式合約」，告訴模型每個欄位的名稱、型別與是否必填，
Responses API 收到後會盡量照這個格式輸出。


| 關鍵字 | 作用 |
|--------|------|
| `type` | 資料型別，例如 `object`、`string`、`integer`、`array` |
| `properties` | 定義每個欄位的名稱與型別 |
| `required` | 哪些欄位一定要出現 |
| `additionalProperties` | 設為 `false`，禁止模型多加欄位 |
| `strict` | Responses API 專屬參數，開啟後嚴格照 schema 輸出 |


In [ ]:
# 手動定義 JSON Schema，並用 Responses API 的 text.format 指定輸出格式
review_schema = {
    "type": "object",
    "properties": {
        "customer_name": {"type": "string"},
        "rating": {"type": "integer"},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "summary": {"type": "string"},
    },
    "required": ["customer_name", "rating", "sentiment", "summary"],
    "additionalProperties": False,
}

response = client.responses.create(
    model=MODEL,
    input=f"請從以下評論抽取顧客姓名、評分（1~5）、情緒與一句話摘要：\n\n{review_text}",
    text={
        "format": {
            "type": "json_schema",
            "name": "review_insight",
            "schema": review_schema,
            "strict": True,
        }
    },
)

data = json.loads(response.output_text)
print(data)
print("型別：", type(data), "｜評分型別：", type(data["rating"]))


---
## 🧬 第四節：用 Pydantic 定義資料模型

手動寫 JSON Schema 容易漏欄位或打錯型別。`client.responses.parse()` 可以直接吃 Pydantic 模型，
自動產生 Schema，並把回覆轉成可以用 `.欄位名` 存取的 Python 物件。


In [ ]:
# 用 Pydantic 定義和上一節相同的資料格式
class ReviewInsight(BaseModel):
    customer_name: str
    rating: int = Field(ge=1, le=5)
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str


print(ReviewInsight.model_json_schema())


In [ ]:
# 用 client.responses.parse() 直接取得 Pydantic 物件
response = client.responses.parse(
    model=MODEL,
    input=f"請從以下評論抽取顧客姓名、評分（1~5）、情緒與一句話摘要：\n\n{review_text}",
    text_format=ReviewInsight,
)

insight = response.output_parsed
print(insight)
print("型別：", type(insight))


---
## 🪆 第五節：巢狀資料與清單（陣列）抽取

真實資料常常是「一個物件裡有一個清單」，例如會議記錄裡有多個待辦事項。
Pydantic 模型可以直接巢狀組合，對應這種結構。


In [ ]:
# 巢狀模型：MeetingMinutes 裡面有一份 ActionItem 清單
class ActionItem(BaseModel):
    owner: str
    task: str
    due_date: Optional[str] = None   # 不一定每個待辦都有期限


class MeetingMinutes(BaseModel):
    meeting_topic: str
    attendees: list[str]
    action_items: list[ActionItem]


In [ ]:
# 示範：從一段會議記錄抽取主題、出席者與待辦清單
meeting_text = """
今天的產品會議由小美主持，出席者有小美、志明、雅婷。
決議事項：
1. 志明要在下週五前完成新版首頁設計。
2. 雅婷負責整理使用者訪談紀錄，沒有明確期限。
3. 小美會發送會議記錄給全體團隊，今天之內完成。
"""

response = client.responses.parse(
    model=MODEL,
    input=f"請從以下會議記錄抽取主題、出席者與待辦事項：\n\n{meeting_text}",
    text_format=MeetingMinutes,
)

minutes = response.output_parsed
print(minutes)


---
## 🛡️ 第六節：欄位驗證與錯誤處理

真實資料常常不完整或模稜兩可，模型抽取也可能失敗或不符合 schema。
用 `try/except` 接住 `ValidationError` 與一般錯誤，程式才不會整個當掉。


In [ ]:
# 示範：對一段模稜兩可的文字做安全抽取
class StrictReview(BaseModel):
    customer_name: str
    rating: int = Field(ge=1, le=5)
    sentiment: Literal["positive", "neutral", "negative"]


def safe_extract_review(text):
    try:
        response = client.responses.parse(
            model=MODEL,
            input=f"請從以下文字抽取顧客姓名、評分（1~5）與情緒：\n\n{text}",
            text_format=StrictReview,
        )
        return response.output_parsed
    except ValidationError as e:
        print("欄位驗證失敗：", e)
        return None
    except Exception as e:
        print("API 呼叫失敗：", e)
        return None


ambiguous_text = "這次體驗普普通通，沒有特別驚艷，但也不算差。"
result = safe_extract_review(ambiguous_text)
print(result)


### 🧾 6-1 常見欄位驗證技巧

| 技巧 | 用途 |
|------|------|
| `Optional[str] = None` | 欄位可能不存在 |
| `Literal["a", "b"]` | 限制只能是固定幾種值（相當於 enum） |
| `Field(ge=1, le=5)` | 數字要落在指定範圍 |
| `list[Model]` | 一個欄位是另一個模型的清單 |
| `ValidationError` | Pydantic 驗證失敗時丟出的例外 |

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>穩定抽取的做法</b></font><br>
  1. 欄位盡量明確（型別、範圍、允許值）。<br>
  2. 用 <code>try/except</code> 分開處理「驗證失敗」與「API 呼叫失敗」。<br>
  3. 失敗時回傳 <code>None</code>，讓呼叫端自行決定要不要重試或提示使用者。
</div>


---
## 🧩 第七節：封裝成可重複使用的抽取函式

把「呼叫 API + 驗證 + 錯誤處理」包成一個共用函式，之後任何抽取任務都能直接重複使用。


In [ ]:
def extract_structured(text, model_cls, role="你是一位嚴謹的資料抽取助手，只依照提供的文字填寫欄位，不要編造。"):
    try:
        response = client.responses.parse(
            model=MODEL,
            instructions=role,
            input=text,
            text_format=model_cls,
        )
        return response.output_parsed
    except ValidationError as e:
        print("欄位驗證失敗：", e)
        return None
    except Exception as e:
        print("API 呼叫失敗：", e)
        return None


demo = extract_structured(review_text, ReviewInsight)
print(demo)


---
## 🧪 第八節：課堂練習

### 📝 練習 A：履歷抽取

定義一個 `ResumeInfo` 模型，包含姓名、學歷、年資與技能清單，
再用 `extract_structured()` 從一段自我介紹文字中抽取出來。

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
resume_text = """
我是林雅婷，畢業於國立成功大學資訊工程學系，
擁有 3 年後端工程師經驗，熟悉 Python、SQL 與 Docker，
目前想應徵資料工程師的職缺。
"""


class ResumeInfo(BaseModel):
    name: str
    education: str
    years_experience: int
    skills: list[str]


info = extract_structured(resume_text, ResumeInfo)
print(info)


---
### 📦 練習 B：訂單抽取（巢狀清單）

定義 `OrderItem` 與 `OrderInfo`（內含 `OrderItem` 清單），
從一段收據文字抽取訂單編號、顧客姓名、商品明細與運費，並計算總金額。


In [ ]:
receipt_text = """
訂單編號 A1023，顧客：陳建宏。
購買項目：
- 藍牙耳機 x2，單價 990 元
- 保護殼 x1，單價 250 元
運費 60 元。
"""


class OrderItem(BaseModel):
    item_name: str
    quantity: int
    unit_price: int


class OrderInfo(BaseModel):
    order_id: str
    customer_name: str
    items: list[OrderItem]
    shipping_fee: int


order = extract_structured(receipt_text, OrderInfo)
print(order)
if order:
    subtotal = sum(item.quantity * item.unit_price for item in order.items)
    print("商品小計：", subtotal, "，加運費後總額：", subtotal + order.shipping_fee)


---
### 🔁 練習 C（選做）：抽取失敗時的重試機制

寫一個 `extract_with_retry()`，失敗時自動重試最多 `retries` 次，
只要有一次成功就回傳結果；全部失敗才回傳 `None`。


In [ ]:
def extract_with_retry(text, model_cls, retries=2):
    for attempt in range(1, retries + 2):
        result = extract_structured(text, model_cls)
        if result is not None:
            return result
        print(f"第 {attempt} 次抽取失敗，重試中...")
    print("已達重試上限，放棄")
    return None


result = extract_with_retry("這段文字資訊不足，可能會抽取失敗。", ReviewInsight, retries=2)
print(result)


---
## ✅ 本週小結

| 技能 | 本週學了什麼 | 下週用在哪裡 |
|------|-------------|-------------|
| 純文字回覆的限制 | 難以穩定解析，格式容易跑掉 | 理解為何要用 Structured Outputs |
| JSON Schema | `type`／`properties`／`required`／`strict` | 理解 Structured Outputs 原理 |
| Pydantic 模型 | 用 class 定義欄位，自動產生 Schema | 簡化開發、型別安全 |
| 巢狀與清單 | 一個模型裡放另一個模型的清單 | 訂單、會議記錄等複雜資料 |
| 驗證與錯誤處理 | `ValidationError`、`try/except` | 穩定的正式環境程式 |
| 抽取函式封裝 | `extract_structured()` 可重複使用 | Week 7 Streamlit 應用整合 |

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>下週預告</b></font><br>
  Week 7 進入 Git 版本控制與 Streamlit 部署，把這幾週寫的程式變成一個真正能上線的網頁應用。
</div>
